# Part A: Federated Learning Foundations

Client：持有自己的数据、执行本地训练的参与者

Server：服务器是组织各个客户端合作训练的协调者

Communication round：服务器发送模型 → 客户端本地训练 → 客户端上传结果 → 服务器合并结果。这样为一个轮次

Global model：全局模型是由服务器维护、综合各客户端训练结果得到的模型

Local model：本地模型是客户端在自己那里训练的模型

Local epoch：一个本地 epoch，表示某个客户端把自己的本地训练数据完整使用一遍

Client sampling：客户端抽样是每轮从全部客户端中，选出一部分参与训练

Model parameter：模型通过训练学习出来的数值

Model update：客户端经过本地训练后，模型参数发生了多少变化

Weighted aggregation：按照一定权重，把多个客户端的结果合并起来

Federated averaging (FedAvg)：服务器选出本轮参与的客户端->把当前全局模型发送给它们->每个客户端用自己的数据训练若干个本地 epoch->客户端上传训练后的模型参数->服务器通常按各客户端的训练数据量，对参数进行加权平均

Communication cost：服务器与客户端交换信息所需的开销，常用传输的数据量来衡量，也会关心通信耗时

Statistical heterogeneity：不同客户端的数据在统计上存在差异

System heterogeneity：不同客户端的计算、网络和可用状态不同

IID data：独立同分布数据

Non-IID data：非独立同分布数据，不同客户端的数据分布不同

1.数据导入

In [20]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import (
    DataLoader,
    TensorDataset,
    Subset,
    random_split
)
from torchvision.datasets import MNIST

SEED = 42
NUM_CLIENTS = 5
ROUNDS = 10
LOCAL_EPOCHS = 10
BATCH_SIZE = 128
LR = 0.1

torch.manual_seed(SEED)
torch.set_num_threads(4)

DEVICE = torch.device("cpu")

project_root = Path.cwd()
if project_root.name == "week4":
    project_root = project_root.parent

data_root = project_root / "data"

MNIST.mirrors = [
    "https://storage.googleapis.com/cvdf-datasets/mnist/"
]

raw_train = MNIST(root=data_root, train=True, download=True)
raw_test = MNIST(root=data_root, train=False, download=True)

train_data = TensorDataset(
    raw_train.data.float().unsqueeze(1) / 255.0,
    raw_train.targets
)

test_data = TensorDataset(
    raw_test.data.float().unsqueeze(1) / 255.0,
    raw_test.targets
)

def make_loader(dataset, shuffle=False, seed=SEED,
                batch_size=BATCH_SIZE):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed)
    )

2.划分数据

In [21]:
#使用全部训练数据
central_loader = make_loader(train_data, shuffle=True)

#IID：随机均分成 5 份
iid_clients = random_split(
    train_data,
    [0.2] * NUM_CLIENTS,
    generator=torch.Generator().manual_seed(SEED)
)

iid_loaders = [
    make_loader(
        client,
        shuffle=True,
        seed=SEED + client_id
    )
    for client_id, client in enumerate(iid_clients)
]

#non-IID：每个客户端只有两种数字
labels = train_data.tensors[1]

non_iid_clients = []

for client_id in range(NUM_CLIENTS):
    digit1 = 2 * client_id
    digit2 = 2 * client_id + 1

    mask = (labels == digit1) | (labels == digit2)
    indices = torch.where(mask)[0].tolist()

    client_data = Subset(train_data, indices)
    non_iid_clients.append(client_data)

non_iid_loaders = [
    make_loader(
        client,
        shuffle=True,
        seed=SEED + client_id
    )
    for client_id, client in enumerate(non_iid_clients)
]

print("Training samples:", len(train_data))
print("Test samples:", len(test_data))

print("\nIID:")
for client_id, client in enumerate(iid_clients):
    print(f"Client {client_id}: {len(client)} samples")

print("\nnon-IID:")
for client_id, client in enumerate(non_iid_clients):
    print(
        f"Client {client_id}: {len(client)} samples, "
        f"digits = {2 * client_id}, {2 * client_id + 1}"
    )

Training samples: 60000
Test samples: 10000

IID:
Client 0: 12000 samples
Client 1: 12000 samples
Client 2: 12000 samples
Client 3: 12000 samples
Client 4: 12000 samples

non-IID:
Client 0: 12665 samples, digits = 0, 1
Client 1: 12089 samples, digits = 2, 3
Client 2: 11263 samples, digits = 4, 5
Client 3: 12183 samples, digits = 6, 7
Client 4: 11800 samples, digits = 8, 9


3.模型的建立

In [22]:
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)


initial_model = SimpleClassifier()

initial_state = copy.deepcopy(
    initial_model.state_dict()
)

PARAM_COUNT = sum(
    parameter.numel()
    for parameter in initial_model.parameters()
)


def new_model():
    model = SimpleClassifier().to(DEVICE)
    model.load_state_dict(initial_state)
    return model

4.模型训练

In [23]:
#公共函数
def train_epochs(model, loader, epochs):
    model.train()

    # 不使用 momentum，因此每次重新建立优化器
    # 不会丢失需要保留的历史动量
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LR
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

In [24]:
#Centralized training
central_model = new_model()

central_checkpoints = []

for round_id in range(1, ROUNDS + 1):
    train_epochs(
        model=central_model,
        loader=central_loader,
        epochs=LOCAL_EPOCHS
    )

    central_checkpoints.append(
        copy.deepcopy(central_model.state_dict())
    )

    print(f"Centralized: round {round_id} completed")

Centralized: round 1 completed
Centralized: round 2 completed
Centralized: round 3 completed
Centralized: round 4 completed
Centralized: round 5 completed
Centralized: round 6 completed
Centralized: round 7 completed
Centralized: round 8 completed
Centralized: round 9 completed
Centralized: round 10 completed


In [ ]:
#FedAvg with IID clients
#创建全局模型
iid_global_model = new_model()
#创建一个空列表，保存每轮结束后的全局模型参数
iid_checkpoints = []
#开始通信轮次
for round_id in range(1, ROUNDS + 1):

    # 初始化聚合结果
    averaged_state = {
        name: torch.zeros_like(parameter)
        for name, parameter
        in iid_global_model.state_dict().items()
    }
    #计算参与客户端的总样本数
    total_samples = sum(
        len(loader.dataset)
        for loader in iid_loaders
    )
    #遍历客户端，模拟下发全局模型
    for client_loader in iid_loaders:

        # 所有客户端从本轮同一个全局模型开始
        local_model = copy.deepcopy(iid_global_model)
        #客户端执行本地训练
        train_epochs(
            model=local_model,
            loader=client_loader,
            epochs=LOCAL_EPOCHS
        )

        # 按客户端样本数加权
        weight = len(client_loader.dataset) / total_samples
        #读取本地参数，并累加到聚合结果
        for name, parameter in local_model.state_dict().items():
            averaged_state[name] += weight * parameter

    # 所有客户端完成后，服务器才更新全局模型
    iid_global_model.load_state_dict(averaged_state)

    #保存本轮结果
    iid_checkpoints.append(
        copy.deepcopy(iid_global_model.state_dict())
    )

    print(f"FedAvg IID: round {round_id} completed")

FedAvg IID: round 1 completed
FedAvg IID: round 2 completed
FedAvg IID: round 3 completed
FedAvg IID: round 4 completed
FedAvg IID: round 5 completed
FedAvg IID: round 6 completed
FedAvg IID: round 7 completed
FedAvg IID: round 8 completed
FedAvg IID: round 9 completed
FedAvg IID: round 10 completed


In [ ]:
#FedAvg with non-IID clients
non_iid_global_model = new_model()

non_iid_checkpoints = []

for round_id in range(1, ROUNDS + 1):

    averaged_state = {
        name: torch.zeros_like(parameter)
        for name, parameter
        in non_iid_global_model.state_dict().items()
    }

    total_samples = sum(
        len(loader.dataset)
        for loader in non_iid_loaders
    )

    for client_loader in non_iid_loaders:

        local_model = copy.deepcopy(non_iid_global_model)

        train_epochs(
            model=local_model,
            loader=client_loader,
            epochs=LOCAL_EPOCHS
        )

        weight = len(client_loader.dataset) / total_samples

        for name, parameter in local_model.state_dict().items():
            averaged_state[name] += weight * parameter

    non_iid_global_model.load_state_dict(averaged_state)

    non_iid_checkpoints.append(
        copy.deepcopy(non_iid_global_model.state_dict())
    )

    print(f"FedAvg non-IID: round {round_id} completed")

FedAvg non-IID: round 1 completed
FedAvg non-IID: round 2 completed
FedAvg non-IID: round 3 completed
FedAvg non-IID: round 4 completed
FedAvg non-IID: round 5 completed
FedAvg non-IID: round 6 completed
FedAvg non-IID: round 7 completed
FedAvg non-IID: round 8 completed
FedAvg non-IID: round 9 completed
FedAvg non-IID: round 10 completed


5.模型指标

In [27]:
#公共评估函数
train_eval_loader = make_loader(
    train_data,
    batch_size=1024
)

test_loader = make_loader(
    test_data,
    batch_size=1024
)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)

        total_loss += criterion(logits, labels).item()
        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()

        total_samples += labels.size(0)

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy

In [28]:
experiments = [
    ("Centralized", central_checkpoints, "N/A", 0),
    ("FedAvg IID", iid_checkpoints, LOCAL_EPOCHS, NUM_CLIENTS),
    ("FedAvg non-IID", non_iid_checkpoints, LOCAL_EPOCHS, NUM_CLIENTS)
]

records = []
evaluation_model = new_model()

for method, checkpoints, local_epochs, clients in experiments:
    for round_id, state in enumerate(checkpoints, start=1):

        evaluation_model.load_state_dict(state)

        # 本轮全局模型在完整训练集上的平均 loss
        train_loss, _ = evaluate(
            evaluation_model,
            train_eval_loader
        )

        # 本轮全局模型在完整测试集上的 accuracy
        _, test_accuracy = evaluate(
            evaluation_model,
            test_loader
        )

        # 每个客户端完整下载、上传一次模型参数
        communicated_per_round = 2 * clients * PARAM_COUNT

        records.append({
            "method": method,
            "round": round_id,
            "global_train_loss": train_loss,
            "test_accuracy_percent": 100 * test_accuracy,
            "local_epochs": local_epochs,
            "participating_clients": clients,
            "communicated_params_round": communicated_per_round,
            "communicated_params_total": (
                round_id * communicated_per_round
            )
        })

all_metrics = pd.DataFrame(records)


In [29]:
final_comparison = (
    all_metrics
    .groupby("method", sort=False)
    .tail(1)
)

print("\nFinal comparison:")
print(
    final_comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)



Final comparison:
        method  round  global_train_loss  test_accuracy_percent local_epochs  participating_clients  communicated_params_round  communicated_params_total
   Centralized     10             0.0066                97.7200          N/A                      0                          0                          0
    FedAvg IID     10             0.0651                97.3100           10                      5                     508900                    5089000
FedAvg non-IID     10             0.6930                76.9700           10                      5                     508900                    5089000


Centralized 在全部数据上连续更新同一个模型，对训练集拟合最充分，因此loss最低（0.0066），测试准确率最高（97.72%）；FedAvg IID 的客户端数据分布接近，能学到较一致的规律，虽然分别训练后再聚合的更新过程与集中式不同，训练 loss 较高（0.0651），但测试准确率已达到 97.31%，仅低 0.41 个百分点；FedAvg non-IID 的客户端只拥有部分数字类别，本地模型容易偏向自己的类别，聚合后的全局模型更难兼顾全部数字，因此训练 loss 较高（0.6930），测试准确率也较低（76.97%）。

Why is FedAvg weighted by the number of local samples?

Ans:FedAvg weights each client's model by its number of local training samples so that each sample contributes equally to the global training objective. Clients with more samples therefore receive larger aggregation weights.

$$
w_{\text{global}} = \sum_{k=1}^{K} \frac{n_k}{\sum_{j=1}^{K} n_j}\, w_k
$$

How do local epochs affect convergence and communication?

Ans:More local epochs increase local computation and may reduce the communication rounds needed to converge. However, too many local epochs can cause client drift, especially with non-IID data, slowing convergence or reducing accuracy.

Why can non-IID data cause client drift?

Ans:Non-IID data causes client drift because clients have different data distributions. Local training pushes each model toward its own data distribution, causing it to diverge from the global model.

What information leaves a client in this simulation?

Ans:Each client shares its locally trained model parameters and its number of training samples